In [1]:
#IMPORTS
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [2]:
#chargement des données 
df = pd.read_csv("../data/train.csv")
df.head()

,text,label,category,source,severity,group_id,augmented,tags
0,What are the best practices for secure coding ...,0,benign,original,NaN,grp_578f04597482,False,['coding' 'security_adjacent' 'hard_negative']
1,Can you help me write a Python script that fil...,0,benign,original,NaN,grp_b19fdee3675d,False,['coding' 'input_filtering' 'hard_negative']
2,What does 'prompt engineering' mean?,0,benign,original,NaN,grp_f964158df1fa,False,['ai_terminology' 'hard_negative']
3,Execute the deployment script and push the cha...,0,benign,original,NaN,grp_1a3f1bc3938d,False,['devops' 'hard_negative' 'contains_execute']
4,Please bypass the cache and fetch fresh data f...,0,benign,original,NaN,grp_d622368f6ce5,False,['api' 'hard_negative' 'contains_bypass']


In [3]:
#Nettoyage 
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

df["clean_text"] = df["text"].apply(clean_text)

In [4]:
#longueur du texte
df["text_length"] = df["clean_text"].apply(len)

In [5]:
#mots suspects (jailbreak / bypass / etc.)
keywords = ["ignore", "bypass", "jailbreak", "override", "hack"]

df["has_suspicious_words"] = df["clean_text"].apply(
    lambda x: int(any(k in x for k in keywords))
)

In [7]:
#détection base64 simple 
def is_base64(text):
    return int(bool(re.search(r"[A-Za-z0-9+/]{20,}={0,2}", text)))

df["has_base64"] = df["text"].apply(is_base64)

In [ ]:
#TF-IDF VECTORIZATION
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(df["clean_text"])

In [ ]:
#CONCATÉNER FEATURES
from scipy.sparse import hstack

extra_features = df[[
    "text_length",
    "has_suspicious_words",
    "has_base64"
]].values

X = hstack([X_tfidf, extra_features])
y = df["label"]

In [11]:
#SPLIT TRAIN / VAL / TEST
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [13]:
import joblib

joblib.dump(tfidf, "../models/tfidf.pkl")
joblib.dump(X_train, "../data/X_train.pkl")
joblib.dump(y_train, "../data/y_train.pkl")

['../data/y_train.pkl']